# Integrating ETL Pipeline with Airflow


## Prerequisites

- **`03_scheduling_and_retries.ipynb`** complete.
- Docker / Compose for Airflow: **[`airflow/README.md`](../../airflow/README.md)**.
- Postgres available (locally or reachable from the Airflow container) with **`users`** table — see **`../../projects/batch_pipeline/schema.sql`**.


## Why integration matters

So far you have:

- **ETL logic** in Python
- **DAGs** in Airflow

Now you **wire them together** so orchestration runs your pipeline code on a schedule with retries and observability.

👉 That combination is how automated production batch systems are built.


## Move ETL code into a script

The course ships **`../../projects/batch_pipeline/etl_pipeline.py`** — extract from JSONPlaceholder, transform fields, load into Postgres.

Mount that folder into the Airflow container (already configured in **`../../airflow/docker-compose.yml`**):

```yaml
volumes:
  - ../projects/batch_pipeline:/opt/airflow/projects/batch_pipeline
```

**Restart** Compose after changing volumes (`docker compose down` then `docker compose up`).

Key functions:

- **`extract()`** — HTTP GET users JSON
- **`transform(data)`** — normalize `name` / `email`
- **`load(data)`** — upsert into **`users`**
- **`run_pipeline()`** — chains all three (single Airflow task)

Connection defaults match **`POSTGRES_*`** env vars (override **`POSTGRES_HOST`** when Postgres is on the host and Airflow runs in Docker — often **`host.docker.internal`** on macOS/Windows). Details: **`../../projects/batch_pipeline/README.md`**.


## Create the Airflow DAG

Repo file: **`../../airflow/dags/etl_pipeline_dag.py`** (`dag_id`: **`etl_pipeline_dag`**).

It adds `projects/batch_pipeline` to **`sys.path`** and calls **`run_pipeline`** from one **`PythonOperator`**.

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

import sys
from pathlib import Path

_BATCH = Path(__file__).resolve().parent.parent / "projects" / "batch_pipeline"
sys.path.insert(0, str(_BATCH))

from etl_pipeline import run_pipeline

default_args = {
    "retries": 2,
    "retry_delay": timedelta(seconds=10),
}

with DAG(
    dag_id="etl_pipeline_dag",
    start_date=datetime(2024, 1, 1),
    schedule_interval="@daily",
    catchup=False,
    default_args=default_args,
) as dag:

    run_etl = PythonOperator(
        task_id="run_etl_pipeline",
        python_callable=run_pipeline,
    )
```

**Multi-task assignment pattern:** see **`../../airflow/dags/etl_pipeline_tasks_dag.py`** (`dag_id`: **`etl_pipeline_tasks_dag`**) — separate **`extract_task` → `transform_task` → `validate_task` → `load_task`** with XCom between steps.


## Run the DAG

1. **`docker compose down`** then **`docker compose up`** from **`airflow/`** if you changed Compose or DAG files.
2. Open **http://localhost:8080**.
3. Enable **`etl_pipeline_dag`** (toggle **On**).
4. **Trigger DAG** manually.
5. Open **`run_etl_pipeline`** → **Logs** and confirm extract / load messages.


## Validate output

In **`psql`** (or any SQL client) against the same database:

```sql
SELECT * FROM users LIMIT 5;
```

You should see normalized **`name`** / **`email`** rows from JSONPlaceholder.


## Logging and monitoring

- **`logging`** in **`etl_pipeline.py`** records pipeline start/end and step context.
- Airflow captures **stdout/stderr** per task try — use the UI **Log** view when debugging.
- For production, add **metrics**, **alerting** (email/Slack/PagerDuty), and central log shipping.

👉 Treat **Airflow task logs** as the first place you look when a run fails.


## Real production flow

**Scheduler (Airflow)** → **runs ETL** → **writes to warehouse/DB** → **downstream analytics / BI**.

👉 Same architecture at small startups and large enterprises — only the operators and infra change.


## Practice

1. Split **`transform`** and **`load`** into **separate `PythonOperator` tasks** (pass rows via **XCom** for small payloads).
2. Compare **single-task** DAG (`etl_pipeline_dag`) vs **multi-task** DAG (`etl_pipeline_tasks_dag`) in the **Graph** view.


## Assignment

Upgrade the pipeline design:

- **Tasks:** **`extract`**, **`transform`**, **`load`** as separate operators
- **Cross-cutting:** retries, structured **logging**, a **validation** step before load
- **Bonus:** simulate a failure (e.g. set **`ETL_SIMULATE_FAILURE=1`** for the Airflow service — see **`etl_pipeline_tasks_dag.py`** / **`load_task`**) and verify retries / alerts behavior in the UI

Reference files: **`../../airflow/dags/etl_pipeline_tasks_dag.py`**, **`../../projects/batch_pipeline/etl_pipeline.py`**.


## Interview questions

1. How do you integrate ETL code with Airflow?
2. Why split work into multiple tasks instead of one big callable?
3. How do you monitor pipelined jobs?
4. How do you debug a failed task instance?


## What you just built

- **DAG** orchestrating **Python ETL**
- **Database** integration with configurable connection
- **Retries** + **logging** aligned with production habits

👉 This mirrors how teams ship batch ingestion every day.


---

**Next:** **Airflow project** — multi-DAG portfolio build (dependencies, production-style scenarios).


**Reality check:** you already know how to build transforms — orchestration is how they become **reliable, scheduled systems** with clear ownership of failures.


## Your tasks

- [ ] Read **`../../projects/batch_pipeline/etl_pipeline.py`** and apply **`schema.sql`** to Postgres.
- [ ] Confirm **`../../airflow/docker-compose.yml`** mounts **`batch_pipeline`** and includes **`_PIP_ADDITIONAL_REQUIREMENTS`** for **`requests`** / **`psycopg2-binary`**.
- [ ] Enable **`etl_pipeline_dag`**, trigger **`run_etl_pipeline`**, validate **`users`** rows in SQL.
- [ ] Optional: run **`etl_pipeline_tasks_dag`** and compare graphs.
